# Court Document Pipeline (Colab GPU)

Downloads court PDFs and extracts text — all on Google's servers using a free GPU.
No files need to leave your laptop. No NVIDIA API quota consumed.

### What this notebook does
1. **Clones the repo** — gets the scraper code and `valid_cases.json` (the list of known case numbers)
2. **Downloads PDFs** from the court API directly into Colab
3. **Extracts text** — PyMuPDF first (free, instant), then Qwen2-VL-7B on GPU for scanned pages
4. **Saves to Google Drive** (persistent) and/or lets you download a zip

### Before you start
- **Runtime → Change runtime type → T4 GPU**
- Have a browser tab open to get a court session ID when prompted

In [ ]:
!pip install -q pymupdf transformers>=4.45 accelerate bitsandbytes qwen-vl-utils Pillow

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected!\n"
        "Go to Runtime > Change runtime type > T4 GPU"
    )

gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU ready: {gpu_name} ({gpu_mem:.1f} GB)")

## Setup

Clone the repo (gets the scraper code + `valid_cases.json`) and optionally
mount Google Drive so downloaded files persist between sessions.

In [ ]:
import os
from pathlib import Path

# ============================================================
# CONFIGURATION — edit these if needed
# ============================================================
REPO_URL = "https://github.com/karimiborna/litigation-outcome-pipeline.git"
BRANCH   = "phase-1/scraper-and-data"

USE_DRIVE = True   # Set False to skip Drive (files won't persist)
DRIVE_DIR = "/content/drive/MyDrive/litigation-pipeline"
# ============================================================

REPO_DIR = "/content/litigation-outcome-pipeline"

if not Path(REPO_DIR).exists():
    !git config --global credential.helper ''
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    print(f"Repo already cloned at {REPO_DIR}")

# Install the project so we can import scraper modules
!pip install -q -e {REPO_DIR}

# Set up output directories
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PDF_DIR    = f"{DRIVE_DIR}/pdfs"
    OUTPUT_DIR = f"{DRIVE_DIR}/extracted"
else:
    PDF_DIR    = f"{REPO_DIR}/data/raw/pdfs"
    OUTPUT_DIR = f"{REPO_DIR}/data/processed/extracted"

os.makedirs(PDF_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"\nPDFs will be saved to:  {PDF_DIR}")
print(f"Text will be saved to: {OUTPUT_DIR}")

## Step 1 — Download PDFs from the court API

Reads `valid_cases.json` from the cloned repo and downloads documents for each case.
You'll need a session ID from the court website — the cell will prompt you.

**To get a session ID:**
1. Open https://webapps.sftc.org/cc/CaseCalendar.dll in your browser
2. Copy the hex string after `SessionID=` in the URL

In [ ]:
import json
import re
import time

import requests

BASE_URL  = "https://webapps.sftc.org"
CASE_PATH = "/ci/CaseInfo.dll"
USER_AGENT = "MSDS603-Research-Scraper/1.0 (SF Small Claims academic study)"
DOWNLOAD_DELAY = 2.5  # seconds between requests


def ask_session_id():
    print("\n" + "=" * 50)
    print("SESSION ID NEEDED")
    print("1. Open https://webapps.sftc.org/cc/CaseCalendar.dll")
    print("2. Copy the hex string after SessionID= from the URL")
    print("=" * 50)
    sid = input("Paste SessionID: ").strip()
    return sid


def get_documents(case_num, session_id):
    url = (
        f"{BASE_URL}{CASE_PATH}"
        f"/datasnap/rest/TServerMethods1/GetDocuments"
        f"/{case_num}/{session_id}/"
    )
    resp = requests.get(url, headers={"User-Agent": USER_AGENT}, timeout=30)
    resp.raise_for_status()
    data = resp.json()

    if data["result"][0] == -1:
        raise Exception("SESSION_EXPIRED")
    if data["result"][0] == 0:
        return []
    return json.loads(data["result"][1])


def sanitize(desc, max_len=40):
    return re.sub(r"[^\w\-]", "_", desc)[:max_len]


# Load valid case numbers from the cloned repo
valid_cases_path = Path(REPO_DIR) / "scraper" / "state" / "valid_cases.json"
with open(valid_cases_path) as f:
    store = json.load(f)
valid_cases = store.get("valid", {})
print(f"Valid cases from repo: {len(valid_cases)}")

# Skip cases whose PDFs are already downloaded
to_download = []
for case_num in sorted(valid_cases.keys()):
    existing = list(Path(PDF_DIR).glob(f"{case_num}_*.pdf"))
    if not existing:
        to_download.append(case_num)

print(f"Already downloaded: {len(valid_cases) - len(to_download)}")
print(f"To download: {len(to_download)}")

if to_download:
    session_id = ask_session_id()
    http = requests.Session()
    http.headers["User-Agent"] = USER_AGENT
    total_pdfs = 0

    for i, case_num in enumerate(to_download):
        try:
            time.sleep(DOWNLOAD_DELAY)
            docs = get_documents(case_num, session_id)
        except Exception as e:
            if "SESSION_EXPIRED" in str(e):
                session_id = ask_session_id()
                time.sleep(DOWNLOAD_DELAY)
                docs = get_documents(case_num, session_id)
            else:
                print(f"  Error for {case_num}: {e}")
                continue

        for doc in docs:
            doc_url = doc.get("URL", "")
            desc = doc.get("DESCRIPTION", "doc")
            if not doc_url:
                continue

            pdf_path = Path(PDF_DIR) / f"{case_num}_{sanitize(desc)}.pdf"
            if pdf_path.exists():
                continue

            time.sleep(DOWNLOAD_DELAY)
            try:
                resp = http.get(doc_url, timeout=60, stream=True)
                if "pdf" in resp.headers.get("content-type", "").lower():
                    with open(pdf_path, "wb") as f:
                        for chunk in resp.iter_content(8192):
                            f.write(chunk)
                    total_pdfs += 1
            except Exception as e:
                print(f"  Download failed: {e}")

        if (i + 1) % 10 == 0:
            print(f"  Progress: {i + 1}/{len(to_download)} cases, {total_pdfs} PDFs")

    print(f"\nDownloaded {total_pdfs} PDFs")
else:
    print("All PDFs already downloaded.")

# List what we have
all_pdfs = sorted(f.name for f in Path(PDF_DIR).glob("*.pdf"))
print(f"\nTotal PDFs on disk: {len(all_pdfs)}")

## Step 2 — PyMuPDF (free text extraction)

Pulls selectable text from each PDF instantly. No GPU needed.
Only PDFs that are scanned images (< 100 chars of text) move to the GPU step.

In [ ]:
import fitz  # PyMuPDF

MIN_TEXT_LENGTH = 100


def extract_with_pymupdf(pdf_path):
    doc = fitz.open(str(pdf_path))
    pages = [page.get_text() for page in doc]
    doc.close()
    return "\n\n".join(pages).strip()


pdfs = sorted(Path(PDF_DIR).glob("*.pdf"))
already_done = []
pymupdf_extracted = []
needs_gpu = []

for pdf_path in pdfs:
    txt_path = Path(OUTPUT_DIR) / f"{pdf_path.stem}.txt"

    if txt_path.exists():
        already_done.append(pdf_path)
        continue

    text = extract_with_pymupdf(pdf_path)
    if len(text) > MIN_TEXT_LENGTH:
        txt_path.write_text(text, encoding="utf-8")
        pymupdf_extracted.append(pdf_path)
    else:
        needs_gpu.append(pdf_path)

print(f"Already extracted (skipped): {len(already_done)}")
print(f"PyMuPDF extracted this run:  {len(pymupdf_extracted)}")
print(f"Need GPU vision model:       {len(needs_gpu)}")

## Step 3 — Load Vision Model

Loads **Qwen2-VL-7B-Instruct** with 4-bit quantization (~4 GB VRAM).
First run downloads ~4 GB of weights (cached for subsequent runs).

Skipped entirely if all PDFs had selectable text.

In [ ]:
model = None
processor = None

if needs_gpu:
    from transformers import (
        AutoProcessor,
        BitsAndBytesConfig,
        Qwen2VLForConditionalGeneration,
    )

    MODEL_ID = "Qwen/Qwen2-VL-7B-Instruct"

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

    print(f"Loading {MODEL_ID} (4-bit quantized)...")
    model = Qwen2VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.bfloat16,
    )
    processor = AutoProcessor.from_pretrained(MODEL_ID)

    used = torch.cuda.memory_allocated() / 1e9
    print(f"Model loaded. GPU memory used: {used:.1f} GB")
else:
    print("All PDFs had selectable text — no model needed.")

## Step 4 — Extract scanned PDFs with GPU

Renders each page to an image and sends it through the vision model.
Uses the same prompt and page-break format as the main repo's extractor.

In [ ]:
import io

from PIL import Image
from qwen_vl_utils import process_vision_info

EXTRACTION_PROMPT = (
    "This is a page from a San Francisco small claims court document. "
    "Extract all text exactly as it appears. Include party names, dates, "
    "claim amounts, rulings, and any other information on the page. "
    "Output plain text only."
)
DPI = 150


def page_to_pil(page, dpi=DPI):
    mat = fitz.Matrix(dpi / 72, dpi / 72)
    pix = page.get_pixmap(matrix=mat, colorspace=fitz.csRGB)
    return Image.open(io.BytesIO(pix.tobytes("png")))


def extract_page(pil_image):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": pil_image},
                {"type": "text", "text": EXTRACTION_PROMPT},
            ],
        }
    ]

    text_input = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text_input],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to("cuda")

    with torch.no_grad():
        output_ids = model.generate(
            **inputs, max_new_tokens=2048, do_sample=False
        )

    generated = output_ids[0, inputs.input_ids.shape[1] :]
    return processor.decode(generated, skip_special_tokens=True)


if needs_gpu and model is not None:
    t0 = time.time()
    for i, pdf_path in enumerate(needs_gpu):
        txt_path = Path(OUTPUT_DIR) / f"{pdf_path.stem}.txt"

        if txt_path.exists():
            continue

        doc = fitz.open(str(pdf_path))
        pages_text = []

        for page_num, page in enumerate(doc, start=1):
            print(
                f"  [{i + 1}/{len(needs_gpu)}] {pdf_path.name}"
                f" - page {page_num}/{len(doc)}",
                flush=True,
            )
            img = page_to_pil(page)
            text = extract_page(img)
            pages_text.append(text)

        doc.close()
        torch.cuda.empty_cache()

        full_text = "\n\n--- PAGE BREAK ---\n\n".join(pages_text)
        txt_path.write_text(full_text, encoding="utf-8")
        print(f"  Saved: {txt_path.name} ({len(full_text):,} chars)")

    elapsed = time.time() - t0
    print(f"\nGPU extraction done: {len(needs_gpu)} PDFs in {elapsed:.0f}s")
else:
    print("No scanned PDFs to process.")

## Results

Summary of all extracted text. Downloads everything as a zip.

In [ ]:
import shutil

txt_files = sorted(Path(OUTPUT_DIR).glob("*.txt"))
total_chars = sum(f.stat().st_size for f in txt_files)

print(f"Total text files: {len(txt_files)}")
print(f"Total size:       {total_chars:,} characters ({total_chars / 1024:.0f} KB)")
print()

for f in txt_files[:20]:
    print(f"  {f.name}  ({f.stat().st_size:,} chars)")
if len(txt_files) > 20:
    print(f"  ... and {len(txt_files) - 20} more")

if USE_DRIVE:
    print(f"\nFiles are saved to Google Drive at: {DRIVE_DIR}")
    print("They will persist even if this runtime disconnects.")

# Create downloadable zip
zip_path = "/content/extracted_texts"
shutil.make_archive(zip_path, "zip", OUTPUT_DIR)
print(f"\nZip: {zip_path}.zip ({Path(zip_path + '.zip').stat().st_size / 1024:.0f} KB)")

from google.colab import files
files.download(f"{zip_path}.zip")